In [ ]:
%load_ext autoreload
%autoreload 2
import os

os.chdir("..")
print(os.getcwd())

In [ ]:
import pandas as pd
import requests
import wandb

In [ ]:
def pprint(dictionary):
    for k, v in dictionary.items():
        print(k, ":", v)

In [ ]:
os.makedirs("outputs/prediction", exist_ok=True)

# Table prep functions

In [ ]:
def format_mean_sem(mean, sem, decimals=3, scale=1.0, math_mode=True):
    """Format a single mean/SEM pair as a LaTeX string, with optional scaling."""
    if pd.isna(mean):
        return "--"
    m = mean * scale
    if pd.isna(sem):
        s = f"{m:.{decimals}f}"
    else:
        s = f"{m:.{decimals}f} \\pm {sem * scale:.{decimals}f}"
    return f"${s}$" if math_mode else s


def add_mean_sem_columns(df, metrics, math_mode=True):
    """
    metrics: dict mapping output_col_name -> dict with:
        mean_col, sem_col, decimals (int), scale (float, default 1.0)
    """
    df = df.copy()
    for out_col, spec in metrics.items():
        df[out_col] = [
            format_mean_sem(
                m,
                s,
                decimals=spec.get("decimals", 3),
                scale=spec.get("scale", 1.0),
                math_mode=math_mode,
            )
            for m, s in zip(df[spec["mean_col"]], df[spec["sem_col"]])
        ]
    return df


def get_latex(df, columns=None, escape=False):
    if columns is not None:
        df = df[columns]
    else:
        columns = list(df.columns)

    latex_table = df.to_latex(
        index=False,
        columns=columns,
        escape=escape,  # False: let \pm and $ pass through untouched
    )
    print(latex_table)

    payload = {
        "formula": latex_table,
        "fsize": "54px",
        "fcolor": "000000",
        "mode": "0",
        "out": "1",
        "remhost": "quicklatex.com",
        "preamble": r"\usepackage{booktabs}\usepackage{amsmath}",
    }
    response = requests.post("https://quicklatex.com/latex3.f", data=payload)
    print(response.text)
    return latex_table


def parse_modality(x):
    for k, v in modalities.items():
        if k in x:
            return v

# WANDB API parsing

In [ ]:
# Connect to wandb api
api = wandb.Api()
entity = "aether_xai"

# S2BMS

In [ ]:
project = "s2bms_prediction"
results_df_path = "outputs/prediction/results.csv"

## csv log

In [ ]:
if os.path.exists(results_df_path):
    df = pd.read_csv(results_df_path)
    print(f"There are {len(df)} records already in results.")
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
else:
    df = None

## fetching

In [ ]:
runs_iterator = api.runs(f"{entity}/{project}")
run_list = []

for i, run in enumerate(runs_iterator):
    if df is not None:
        if run.id in list(df.run_id):
            print(
                f'{run.summary["experiment"]} with seed={run.config["seed"]} already in results.'
            )
            continue
    elif run.state != "finished":
        print(f"{run.id} is not finished.")
        continue

    captures = dict(run.summary)
    captures["run_id"] = run.id
    captures["seed"] = run.config["seed"]
    run_list.append(captures)
    print(f'{run.summary["experiment"]} with seed={run.config["seed"]} logged.')

In [ ]:
runs_df = pd.DataFrame(run_list)
if df is not None:
    runs_df = pd.concat([df, runs_df], ignore_index=True)
runs_df.to_csv("outputs/prediction/results.csv", index=False)

In [ ]:
cols_of_interest = {
    "experiment": "Modality (best config.)",
    # "best_val_loss": "Validation loss",
    "test_mse_loss": "Test MSE loss",
    # "test_top_1_acc": "Test Top-1 acc",
    "test_top_5_acc": "Test Top-5 acc",
    # "train_mse_loss": "Train MSE loss",
    "test_top_10_acc": "Test Top-10 acc",
    # "best_val_mse_loss": "Validation MSE loss",
    # "best_val_top_1_acc": "Validation Top-1 acc",
    # "best_val_top_5_acc": "Validation Top-5 acc",
    # "best_val_top_10_acc": "Validation Top-10 acc",
}

modalities = {
    "aef": "AlphaEarth",
    "tessera": "Tessera",
    "s2": "Sentinel-2",
    "baseline_mlp": "Baseline-MLP",
    "baseline_lin": "Baseline-Lin",
    "geoclip": "GeoCLIP",
    "satclip": "SatCLIP",
}

In [ ]:
sub_runs_df = runs_df[cols_of_interest.keys()]
for col in cols_of_interest.keys():
    if "top" in col:
        sub_runs_df[col] = sub_runs_df[col].apply(lambda x: x * 100)

grouped = sub_runs_df.groupby("experiment")

# mean and SEM in one go
summary_mean = grouped.mean(numeric_only=True)
summary_sem = grouped.sem(numeric_only=True)  # pandas has this built in: std/sqrt(n)

# also useful to know how many runs went into each SEM
summary_n = grouped.size().rename("n_runs")

# rename sem columns so they don't clash with mean columns
summary_sem = summary_sem.rename(columns={c: f"{c}_sem" for c in summary_sem.columns})

summary = summary_mean.join(summary_sem).join(summary_n)
summary = summary.sort_values("test_mse_loss")
summary.reset_index(inplace=True)

In [ ]:
summary["modality"] = summary["experiment"].apply(lambda x: parse_modality(x))
cols_of_interest["modality"] = "Modality (best config.)"

In [ ]:
summary

# Table 1

In [ ]:
# selection_col = 'test_top_5_acc'
# selection_col = "test_top_10_acc"
# selection_col = 'best_val_mse_loss'
# selection_mode = "max"

selection_col = "test_mse_loss"
selection_mode = "min"

if selection_mode == "max":
    tab_1 = summary.loc[summary.groupby("modality")[selection_col].idxmax()]
    tab_1 = tab_1.sort_values(selection_col, ascending=False)
else:
    tab_1 = summary.loc[summary.groupby("modality")[selection_col].idxmin()]
    tab_1 = tab_1.sort_values(selection_col, ascending=True)

# map display column -> (mean_col, sem_col) present in `summary`
metrics = {
    "Test Top-10 acc": {
        "mean_col": "test_top_10_acc",
        "sem_col": "test_top_10_acc_sem",
        "decimals": 1,
    },
    "Test Top-5 acc": {
        "mean_col": "test_top_5_acc",
        "sem_col": "test_top_5_acc_sem",
        "decimals": 1,
    },
    "Test MSE loss [$1e{-2}$]": {
        "mean_col": "test_mse_loss",
        "sem_col": "test_mse_loss_sem",
        "decimals": 2,
        "scale": 100.0,
    },
}

tab_1_fmt = add_mean_sem_columns(tab_1, metrics)
tab_1_fmt = tab_1_fmt.rename(columns={"modality": "Modality (best config.)"})

get_latex(
    tab_1_fmt,
    columns=["Modality (best config.)"] + list(metrics.keys()),
)

In [ ]:
keep = [
    "Modality (best config.)",
    "test_mse_loss",
    # "best_val_loss",
    # "test_top_1_acc",
    "test_top_5_acc",
    # "train_mse_loss",
    "test_top_10_acc",
    # "best_val_mse_loss",
    # "best_val_top_1_acc",
    # "best_val_top_5_acc",
    # "best_val_top_10_acc",
]
tab_1_fmt[keep]

# Table S1 per modality

In [ ]:
for m in modalities.values():
    tab_s1 = summary[summary["modality"] == m]
    if selection_mode == "max":
        tab_s1.sort_values(selection_col, ascending=False, inplace=True)
    else:
        tab_s1.sort_values(selection_col, ascending=True, inplace=True)

    tab_s1

In [ ]:
tab_s1 = summary[summary["modality"] == "Sentinel-2"]
if selection_mode == "max":
    tab_s1.sort_values(selection_col, ascending=False, inplace=True)
else:
    tab_s1.sort_values(selection_col, ascending=True, inplace=True)
# keep = ['modality', 'best_val_loss', 'test_mse_loss', 'test_top_1_acc', 'test_top_5_acc', 'train_mse_loss',  'best_val_mse_loss', 'best_val_top_1_acc', 'best_val_top_5_acc', 'best_val_top_10_acc']
get_latex(
    tab_s1,
    columns=["experiment", "test_top_10_acc", "test_top_5_acc", "test_top_1_acc", "test_mse_loss"],
)
tab_s1

In [ ]:
summary["modality"] = summary["experiment"].apply(lambda x: parse_modality(x))
cols_of_interest["modality"] = "Modality (best config.)"

# Satbird

In [ ]:
project = "satbird-usa-summer_prediction"
results_df_path = "outputs/prediction/results_satbird.csv"

## csv log

In [ ]:
if os.path.exists(results_df_path):
    df = pd.read_csv(results_df_path)
    print(f"There are {len(df)} records already in results.")
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
else:
    df = None

## fetching

In [ ]:
runs_iterator = api.runs(f"{entity}/{project}")
run_list = []

for i, run in enumerate(runs_iterator):
    if df is not None:
        if run.id in list(df.run_id):
            print(
                f'{run.summary["experiment"]} with seed={run.config["seed"]} already in results.'
            )
            continue
    elif run.state != "finished":
        print(f"{run.id} is not finished.")
        continue

    captures = dict(run.summary)
    captures["run_id"] = run.id
    captures["seed"] = run.config["seed"]
    run_list.append(captures)
    print(f'{run.summary["experiment"]} with seed={run.config["seed"]} logged.')

In [ ]:
runs_df = pd.DataFrame(run_list)
if df is not None:
    runs_df = pd.concat([df, runs_df], ignore_index=True)
runs_df.to_csv("outputs/prediction/results.csv", index=False)

In [ ]:
cols_of_interest = {
    "experiment": "Modality (best config.)",
    "test_mse_loss": "Test MSE loss",
    "test_mae_loss": "Test MAE loss",
    "test_top_30_acc": "Test Top-30 acc",
    "test_top_10_acc": "Test Top-10 acc",
}

modalities = {
    "aef": "AlphaEarth",
    "baseline_mlp": "Baseline-MLP",
    "baseline_lin": "Baseline-Lin",
    "geoclip": "GeoCLIP",
    "satclip": "SatCLIP",
}

In [ ]:
sub_runs_df = runs_df[cols_of_interest.keys()]
for col in cols_of_interest.keys():
    if "top" in col:
        sub_runs_df[col] = sub_runs_df[col].apply(lambda x: x * 100)

grouped = sub_runs_df.groupby("experiment")

# mean and SEM in one go
summary_mean = grouped.mean(numeric_only=True)
summary_sem = grouped.sem(numeric_only=True)  # pandas has this built in: std/sqrt(n)

# also useful to know how many runs went into each SEM
summary_n = grouped.size().rename("n_runs")

# rename sem columns so they don't clash with mean columns
summary_sem = summary_sem.rename(columns={c: f"{c}_sem" for c in summary_sem.columns})

summary = summary_mean.join(summary_sem).join(summary_n)
summary = summary.sort_values("test_mse_loss")
summary.reset_index(inplace=True)

In [ ]:
summary["modality"] = summary["experiment"].apply(lambda x: parse_modality(x))
cols_of_interest["modality"] = "Modality (best config.)"

In [ ]:
summary

# Table 1

In [ ]:
# selection_col = 'test_top_5_acc'
# selection_col = "test_top_10_acc"
# selection_col = 'best_val_mse_loss'
# selection_mode = "max"

selection_col = "test_mse_loss"
selection_mode = "min"

if selection_mode == "max":
    tab_1 = summary.loc[summary.groupby("modality")[selection_col].idxmax()]
    tab_1 = tab_1.sort_values(selection_col, ascending=False)
else:
    tab_1 = summary.loc[summary.groupby("modality")[selection_col].idxmin()]
    tab_1 = tab_1.sort_values(selection_col, ascending=True)

# map display column -> (mean_col, sem_col) present in `summary`
metrics = {
    "Test Top-10 acc": {
        "mean_col": "test_top_10_acc",
        "sem_col": "test_top_10_acc_sem",
        "decimals": 1,
    },
    "Test Top-30 acc": {
        "mean_col": "test_top_30_acc",
        "sem_col": "test_top_30_acc_sem",
        "decimals": 1,
    },
    "Test MSE loss [$1e{-2}$]": {
        "mean_col": "test_mse_loss",
        "sem_col": "test_mse_loss_sem",
        "decimals": 2,
        "scale": 100.0,
    },
    "Test MAE loss [$1e{-2}$]": {
        "mean_col": "test_mae_loss",
        "sem_col": "test_mae_loss_sem",
        "decimals": 2,
        "scale": 100.0,
    },
}

tab_1_fmt = add_mean_sem_columns(tab_1, metrics)
tab_1_fmt = tab_1_fmt.rename(columns={"modality": "Modality (best config.)"})

get_latex(
    tab_1_fmt,
    columns=["Modality (best config.)"] + list(metrics.keys()),
)

# Table S1 per modality

In [ ]:
keep = [
    "Modality (best config.)",
    "test_mse_loss",
    "test_mae_loss",
    "test_top_10_acc",
    "test_top_30_acc",
]
tab_1_fmt[keep]

In [ ]:
for m in modalities.values():
    tab_s1 = summary[summary["modality"] == m]
    if selection_mode == "max":
        tab_s1.sort_values(selection_col, ascending=False, inplace=True)
    else:
        tab_s1.sort_values(selection_col, ascending=True, inplace=True)

    tab_s1

In [ ]:
tab_s1 = summary[summary["modality"] == "AlphaEarth"]
if selection_mode == "max":
    tab_s1.sort_values(selection_col, ascending=False, inplace=True)
else:
    tab_s1.sort_values(selection_col, ascending=True, inplace=True)
# keep = ['modality', 'best_val_loss', 'test_mse_loss', 'test_top_1_acc', 'test_top_5_acc', 'train_mse_loss',  'best_val_mse_loss', 'best_val_top_1_acc', 'best_val_top_5_acc', 'best_val_top_10_acc']
get_latex(
    tab_s1,
    columns=["experiment", "test_top_10_acc", "test_top_30_acc", "test_mse_loss", "test_mae_loss"],
)
tab_s1